In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import time

# Chunk dosyasını yükle
with open('/content/drive/MyDrive/LegalRAG/mevzuat_chunked0v2_normalized.json', 'r') as f:
    chunks = json.load(f)

print(f"Toplam chunk: {len(chunks)}")

def soru_uret(chunk, n=3):
    prompt = f"""Aşağıdaki kanun maddesinden {n} farklı soru üret.

Kurallar:
- Günlük konuşma dilinde yaz
- Madde numarası kullanma
- Hukuki terim kullanmaktan kaçın
- Sorular birbirinden farklı olsun
- Sonraki başlığı da göz önünde bulundur

Kanun: {chunk['metadata']['kanun']}
Madde: {chunk['metadata']['madde_no']}
İçerik: {chunk['text'][:500]}
Sonraki başlık: {chunk['metadata'].get('sonraki_baslik', '')}

Sadece JSON formatında döndür, başka hiçbir şey yazma:
[{{"soru": "..."}}, {{"soru": "..."}}, {{"soru": "..."}}]"""

    try:
        message = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
        response = message.content[0].text.strip()

        # Markdown code block temizle
        response = response.replace('```json', '').replace('```', '').strip()

        sorular = json.loads(response)
        return [s['soru'] for s in sorular]
    except Exception as e:
        print(f"Hata: {e}")
        return []

# Test
print("\n=== TEST ===")
for chunk in chunks[:3]:
    sorular = soru_uret(chunk)
    print(f"\n{chunk['metadata']['madde_no']}:")
    for s in sorular:
        print(f"  - {s}")
    time.sleep(0.5)


In [ ]:
import json
import time

embedding_train = []
gold_test = []
hatalar = []

print(f"Toplam chunk: {len(chunks)}")
print("Üretim başlıyor...\n")

for i, chunk in enumerate(chunks):
    sorular = soru_uret(chunk, n=3)

    if len(sorular) < 3:
        hatalar.append(i)
        print(f"Hata - Chunk {i}: {chunk['metadata']['madde_no']}")
        continue

    chunk_id = chunk['metadata']['chunk_id']
    kanun = chunk['metadata']['kanun']
    madde_no = chunk['metadata']['madde_no']

    # İlk 2 soru → embedding FT train
    for soru in sorular[:2]:
        embedding_train.append({
            "soru": soru,
            "chunk_id": chunk_id,
            "kanun": kanun,
            "madde_no": madde_no
        })

    # 3. soru → gold test
    gold_test.append({
        "soru": sorular[2],
        "chunk_id": chunk_id,
        "kanun": kanun,
        "madde_no": madde_no
    })

    # Her 50 chunk'ta bir kaydet
    if (i + 1) % 50 == 0:
        print(f"İlerleme: {i+1}/{len(chunks)}")

        with open('/content/drive/MyDrive/LegalRAG/data/embedding_train_synthetic.json', 'w') as f:
            json.dump(embedding_train, f, ensure_ascii=False, indent=2)

        with open('/content/drive/MyDrive/LegalRAG/data/gold_test_synthetic.json', 'w') as f:
            json.dump(gold_test, f, ensure_ascii=False, indent=2)

    time.sleep(0.3)

# Final kaydet
with open('/content/drive/MyDrive/LegalRAG/data/embedding_train_synthetic.json', 'w') as f:
    json.dump(embedding_train, f, ensure_ascii=False, indent=2)

with open('/content/drive/MyDrive/LegalRAG/data/gold_test_synthetic.json', 'w') as f:
    json.dump(gold_test, f, ensure_ascii=False, indent=2)

print(f"\n✅ Tamamlandı!")
print(f"Embedding train: {len(embedding_train)} soru")
print(f"Gold test: {len(gold_test)} soru")
print(f"Hatalı chunk: {len(hatalar)}")

In [ ]:
print(f"Hatalı chunk indexleri: {hatalar}")

# Hatalıları tekrar üret
for i in hatalar:
    chunk = chunks[i]
    sorular = soru_uret(chunk, n=3)

    if len(sorular) >= 3:
        chunk_id = chunk['metadata']['chunk_id']
        kanun = chunk['metadata']['kanun']
        madde_no = chunk['metadata']['madde_no']

        embedding_train.append({"soru": sorular[0], "chunk_id": chunk_id, "kanun": kanun, "madde_no": madde_no})
        embedding_train.append({"soru": sorular[1], "chunk_id": chunk_id, "kanun": kanun, "madde_no": madde_no})
        gold_test.append({"soru": sorular[2], "chunk_id": chunk_id, "kanun": kanun, "madde_no": madde_no})
        print(f"✅ Chunk {i} tamamlandı: {madde_no}")
    else:
        print(f"❌ Chunk {i} yine hatalı: {chunk['metadata']['madde_no']}")

    time.sleep(1)

# Kaydet
with open('/content/drive/MyDrive/LegalRAG/data/embedding_train_synthetic.json', 'w') as f:
    json.dump(embedding_train, f, ensure_ascii=False, indent=2)

with open('/content/drive/MyDrive/LegalRAG/data/gold_test_synthetic.json', 'w') as f:
    json.dump(gold_test, f, ensure_ascii=False, indent=2)

print(f"\nFinal - Embedding train: {len(embedding_train)}")
print(f"Final - Gold test: {len(gold_test)}")     // Bu gold testi kullanmadık. Train olarak da görülmedi.

In [ ]:
!pip install sentence-transformers faiss-gpu numpy -q

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import faiss
import numpy as np
import json

# Verileri yükle
with open('/content/drive/MyDrive/LegalRAG/data/embedding_train_synthetic.json', 'r') as f:
    train_data = json.load(f)

with open('/content/drive/MyDrive/LegalRAG/data/gold_test_synthetic.json', 'r') as f:
    gold_test = json.load(f)

with open('/content/drive/MyDrive/LegalRAG/data/mevzuat_chunked0v2_normalized.json', 'r') as f:
    chunks = json.load(f)

print(f"Train: {len(train_data)} soru")
print(f"Gold test: {len(gold_test)} soru")
print(f"Chunk: {len(chunks)} chunk")

# Chunk_id → index mapping
chunk_id_to_idx = {c['metadata']['chunk_id']: i for i, c in enumerate(chunks)}
chunk_texts = [c['text'] for c in chunks]

print(f"\nChunk mapping hazır")
print(f"Örnek chunk_id: {list(chunk_id_to_idx.keys())[0]}")

In [ ]:
# Ham Mursit-Large yükle
print("Model yükleniyor...")
model = SentenceTransformer("newmindai/Mursit-Large-TR-Retrieval")
print(f"Model boyutu: {model.get_sentence_embedding_dimension()}")

# Chunk'ları encode et
print("\nChunk'lar encode ediliyor...")
chunk_embeddings = model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Embedding shape: {chunk_embeddings.shape}")

# FAISS index kur
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"FAISS index kuruldu: {index.ntotal} vektör")

In [ ]:
def metrikleri_olc(model, index, gold_test, chunk_id_to_idx, k=10):
    sorular = [s['soru'] for s in gold_test]
    dogru_chunk_idler = [s['chunk_id'] for s in gold_test]

    # Soruları encode et
    print("Sorular encode ediliyor...")
    soru_embeddings = model.encode(
        sorular,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    # FAISS'te ara
    print("FAISS araması yapılıyor...")
    scores, indices = index.search(soru_embeddings, k)

    recall_5 = 0
    recall_10 = 0
    mrr = 0
    ndcg = 0

    for i, dogru_id in enumerate(dogru_chunk_idler):
        dogru_idx = chunk_id_to_idx.get(dogru_id, -1)
        if dogru_idx == -1:
            continue

        top_k = indices[i].tolist()

        # Recall@5
        if dogru_idx in top_k[:5]:
            recall_5 += 1

        # Recall@10
        if dogru_idx in top_k[:10]:
            recall_10 += 1

        # MRR
        if dogru_idx in top_k:
            rank = top_k.index(dogru_idx) + 1
            mrr += 1 / rank

        # nDCG@10
        if dogru_idx in top_k:
            rank = top_k.index(dogru_idx) + 1
            import math
            ndcg += 1 / math.log2(rank + 1)

    n = len(gold_test)
    return {
        "Recall@5":  round(recall_5 / n, 4),
        "Recall@10": round(recall_10 / n, 4),
        "MRR":       round(mrr / n, 4),
        "nDCG@10":   round(ndcg / n, 4)
    }

# Baseline ölç
print("=== BASELINE (Ham Mursit-Large) ===")
baseline_metrikler = metrikleri_olc(model, index, gold_test, chunk_id_to_idx)
for metrik, deger in baseline_metrikler.items():
    print(f"{metrik}: {deger}")

In [ ]:
import json, torch, numpy as np, faiss, math, gc
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from torch.utils.data import DataLoader
from tqdm import tqdm

with open('/content/drive/MyDrive/LegalRAG/data/embedding_train_synthetic.json', 'r') as f:
    train_data = json.load(f)

with open('/content/drive/MyDrive/LegalRAG/data/mevzuat_chunked0v2_normalized.json', 'r') as f:
    chunks = json.load(f)

chunk_id_to_idx = {c['metadata']['chunk_id']: i for i, c in enumerate(chunks)}
chunk_texts = [c['text'] for c in chunks]
print(f"Hazır. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Önce GPU belleğini temizle
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

# Batch size küçült
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=8  # 32'den 8'e düşür
)

# Eğitim
print("\nTur 1 eğitimi başlıyor...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    show_progress_bar=True,
    output_path='/content/drive/MyDrive/LegalRAG/models/mursit_large_tur1'
    use_amp=True
)

print("✅ Tur 1 tamamlandı")    # Training loss : 0.0145


In [ ]:
# Hard negative mining
print("Model yükleniyor...")
model = SentenceTransformer('/content/drive/MyDrive/LegalRAG/models/mursit_large_tur1')

chunk_emb = model.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)

sorular = [s['soru'] for s in train_data]
dogru_ids = [s['chunk_id'] for s in train_data]
soru_emb = model.encode(sorular, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
_, indices = index.search(soru_emb, 20)

hard_neg_data = []
for i, (soru, dogru_id) in enumerate(zip(sorular, dogru_ids)):
    dogru_idx = chunk_id_to_idx.get(dogru_id, -1)
    if dogru_idx == -1: continue
    hard_negs = [idx for idx in indices[i].tolist() if idx != dogru_idx][:5]
    if len(hard_negs) < 5: continue
    hard_neg_data.append({
        "soru": soru,
        "dogru_chunk": chunk_texts[dogru_idx],
        "hard_negatives": [chunk_texts[idx] for idx in hard_negs]
    })

print(f"Hard negative: {len(hard_neg_data)} örnek")

# GPU temizle
del model, chunk_emb, soru_emb, indices, index
torch.cuda.empty_cache()
gc.collect()
print(f"Temizlendi. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Tur 2 — CachedMultipleNegativesRankingLoss
train_examples = []
for item in hard_neg_data:
    texts = [item['soru'], item['dogru_chunk']] + item['hard_negatives']
    train_examples.append(InputExample(texts=texts))

print(f"Toplam: {len(train_examples)} örnek")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

model_tur2 = SentenceTransformer('/content/drive/MyDrive/LegalRAG/models/mursit_large_tur1')
train_loss = CachedMultipleNegativesRankingLoss(model_tur2, mini_batch_size=32)

print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model_tur2.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=50,
    show_progress_bar=True,
    use_amp=True,
    output_path='/content/drive/MyDrive/LegalRAG/models/mursit_large_tur2_v2'
)

print("✅ Tur 2 tamamlandı")

In [ ]:
import math

def metrikleri_olc(model, index, chunk_id_to_idx, k=10):
    with open('/content/drive/MyDrive/LegalRAG/data/gold_test_synthetic.json', 'r') as f:
        gold_test = json.load(f)

    sorular = [s['soru'] for s in gold_test]
    dogru_chunk_idler = [s['chunk_id'] for s in gold_test]

    soru_emb = model.encode(sorular, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    _, indices = index.search(soru_emb, k)

    recall_5 = recall_10 = mrr = ndcg = 0
    for i, dogru_id in enumerate(dogru_chunk_idler):
        dogru_idx = chunk_id_to_idx.get(dogru_id, -1)
        if dogru_idx == -1: continue
        top_k = indices[i].tolist()
        if dogru_idx in top_k[:5]:  recall_5 += 1
        if dogru_idx in top_k[:10]: recall_10 += 1
        if dogru_idx in top_k:
            rank = top_k.index(dogru_idx) + 1
            mrr += 1 / rank
            ndcg += 1 / math.log2(rank + 1)

    n = len(gold_test)
    return {
        "Recall@5":  round(recall_5/n, 4),
        "Recall@10": round(recall_10/n, 4),
        "MRR":       round(mrr/n, 4),
        "nDCG@10":   round(ndcg/n, 4)
    }

# Tur 2 modelini değerlendir
chunk_emb_tur2 = model_tur2.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
index_tur2 = faiss.IndexFlatIP(chunk_emb_tur2.shape[1])
index_tur2.add(chunk_emb_tur2)

print("=== LARGE TUR 2 v2 METRİKLERİ ===")
tur2_v2 = metrikleri_olc(model_tur2, index_tur2, chunk_id_to_idx)
for m, v in tur2_v2.items():
    print(f"{m}: {v}")

print("\n=== TÜM SONUÇLAR ===")
print(f"{'Metrik':<12} {'Base Base':>10} {'Base Tur1':>10} {'Large Tur1':>11} {'Large Tur2':>11}")
results = {
    "Base Base":  {"Recall@5": 0.6966, "Recall@10": 0.7724, "MRR": 0.5365, "nDCG@10": 0.5934},
    "Base Tur1":  {"Recall@5": 0.8602, "Recall@10": 0.9157, "MRR": 0.7107, "nDCG@10": 0.7608},
    "Large Tur1": {"Recall@5": 0.8791, "Recall@10": 0.9324, "MRR": 0.7376, "nDCG@10": 0.7853},
}
for m in ["Recall@5", "Recall@10", "MRR", "nDCG@10"]:
    print(f"{m:<12} {results['Base Base'][m]:>10.4f} {results['Base Tur1'][m]:>10.4f} {results['Large Tur1'][m]:>11.4f} {tur2_v2[m]:>11.4f}")

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

# Tur 2 modeli ile hard negative mining
print("Hard negative mining başlıyor...")
chunk_emb = model_tur2.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)

sorular = [s['soru'] for s in train_data]
dogru_ids = [s['chunk_id'] for s in train_data]
soru_emb = model_tur2.encode(sorular, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
_, indices = index.search(soru_emb, 20)

hard_neg_tur3 = []
for i, (soru, dogru_id) in enumerate(zip(sorular, dogru_ids)):
    dogru_idx = chunk_id_to_idx.get(dogru_id, -1)
    if dogru_idx == -1: continue
    hard_negs = [idx for idx in indices[i].tolist() if idx != dogru_idx][:5]
    if len(hard_negs) < 5: continue
    hard_neg_tur3.append({
        "soru": soru,
        "dogru_chunk": chunk_texts[dogru_idx],
        "hard_negatives": [chunk_texts[idx] for idx in hard_negs]
    })

print(f"Hard negative: {len(hard_neg_tur3)} örnek")

# GPU temizle
del chunk_emb, soru_emb, indices, index
torch.cuda.empty_cache()
gc.collect()
print(f"Temizlendi. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Tur 3 eğitimi
train_examples_tur3 = []
for item in hard_neg_tur3:
    texts = [item['soru'], item['dogru_chunk']] + item['hard_negatives']
    train_examples_tur3.append(InputExample(texts=texts))

train_dataloader_tur3 = DataLoader(train_examples_tur3, shuffle=True, batch_size=32)

model_tur3 = SentenceTransformer('/content/drive/MyDrive/LegalRAG/models/mursit_large_tur2_v2')
train_loss = CachedMultipleNegativesRankingLoss(model_tur3, mini_batch_size=32)

print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model_tur3.fit(
    train_objectives=[(train_dataloader_tur3, train_loss)],
    epochs=2,
    warmup_steps=50,
    show_progress_bar=True,
    use_amp=True,
    output_path='/content/drive/MyDrive/LegalRAG/models/mursit_large_tur3'
)

print("✅ Tur 3 tamamlandı")

In [ ]:
chunk_emb_tur3 = model_tur3.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
index_tur3 = faiss.IndexFlatIP(chunk_emb_tur3.shape[1])
index_tur3.add(chunk_emb_tur3)

print("=== LARGE TUR 3 METRİKLERİ ===")
tur3 = metrikleri_olc(model_tur3, index_tur3, chunk_id_to_idx)
for m, v in tur3.items():
    print(f"{m}: {v}")

print("\n=== TÜM SONUÇLAR ===")
print(f"{'Metrik':<12} {'Baseline':>10} {'Tur 1':>10} {'Tur 2':>10} {'Tur 3':>10}")
results = {
    "Baseline": {"Recall@5": 0.7341, "Recall@10": 0.8206, "MRR": 0.5752, "nDCG@10": 0.6345},
    "Tur 1":    {"Recall@5": 0.8791, "Recall@10": 0.9324, "MRR": 0.7376, "nDCG@10": 0.7853},
    "Tur 2":    {"Recall@5": 0.8868, "Recall@10": 0.9329, "MRR": 0.7575, "nDCG@10": 0.8005},
}
for m in ["Recall@5", "Recall@10", "MRR", "nDCG@10"]:
    print(f"{m:<12} {results['Baseline'][m]:>10.4f} {results['Tur 1'][m]:>10.4f} {results['Tur 2'][m]:>10.4f} {tur3[m]:>10.4f}")